# 04 - Reading the bias sweep

**Purpose.** To explain what session 01 actually found, slowly and in order: what was captured,
what each published number means, how it was arrived at, and what it now lets the project do that
it could not do before. `03` is the notebook that *made* these numbers and is written for someone
checking the work. This one is written for someone deciding what to do next.

**What it is not for.** It measures nothing and writes nothing. Every number below is read back
from `results/` - `bias_sweep.csv`, `bias_constants.json`, `pedestal_drift.csv` - or, in two
places where a picture needs pixels, from the session's own frames. If a number here disagreed
with `results/`, `results/` would be right and this notebook would be the bug.

**It assumes `00_statistics.ipynb`.** Every statistical move - spatial versus temporal spread,
why noise is measured from a difference of two frames, what an uncertainty is, why a fraction
beats a minimum - is explained there on these same frames. This notebook cites it rather than
re-deriving it.

**The units, once.** Everything is in **ADC counts**, where full scale is 4095. The FITS files
store 16x that, and `stats.to_adc` is the only conversion. No electrons appear anywhere: that
needs the system gain `g`, and `g` is the photon transfer curve, which has not run. A read noise
in electrons quoted today would have a made-up number in it.

## What session 01 was for

Four questions, agreed before a frame was shot, each pinning down a term the SNR model needs:

| question | why the model needs it |
|---|---|
| what is the **pedestal**, at every gain and offset? | it is subtracted from every frame this project ever calibrates |
| what is the **read noise** `R`, at every gain? | it is the `R^2 / t` term - the only noise that a longer sub-exposure dilutes |
| where is the **HCG threshold**? | it is a step change in `R` for free, and it decides which gains are worth using |
| what **offset** should the project fix on? | too low clips the data; too high wastes full scale |

And one question that was really about session 02: **does the pedestal drift**, and therefore
must darks be interleaved with bias frames?

In [ ]:
import json, math, pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropix import fits as F, spatial as SP, stats as ST

pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "font.size": 8})

RESULTS = pathlib.Path("..") / "results"
FRAMES = pathlib.Path("..") / "data" / "session01" / "frames"
SPECS = pathlib.Path("..") / "vendor" / "asi585specs" / "gain-curves.csv"

sweep = pd.read_csv(RESULTS / "bias_sweep.csv")
drift = pd.read_csv(RESULTS / "pedestal_drift.csv")
with open(RESULTS / "bias_constants.json") as fh:
    K = json.load(fh)

PROJECT_OFFSET = K["project_offset"]["value"]
HCG = K["hcg_threshold_gain"]["value"]
at15 = sweep[sweep.offset == PROJECT_OFFSET].sort_values("gain")

# Section 6 needs pixels.  Everything else reads the published tables.
HAVE_FRAMES = FRAMES.is_dir()
if not HAVE_FRAMES:
    print(f"note: {FRAMES} is absent, so section 6's pixel panel is skipped.\n"
          "Every other section reads results/ and is unaffected.")

print(f"{len(sweep)} settings in bias_sweep.csv, "
      f"{K['project_offset']['source_frames']:,} frames behind them")
print(f"{len(K)} published constants:")
for name, c in K.items():
    v = c["value"]
    v = "per-branch fit" if isinstance(v, dict) else f"{v:g}"
    print(f"  {name:<26} {v:>16}  {c['unit']}")

---

## 1. What was actually captured

2,790 frames at 157 distinct `(gain, offset)` settings, all at -10 C, all at the camera's
minimum exposure of 32 microseconds - short enough that no light and no dark current can
accumulate, so what is left is the pedestal plus the read noise, which is exactly the pair we
want to isolate.

The sampling is deliberately uneven, and the shape of it is an argument:

- **the coarse block** walks gain 0 to 600 in steps of 10 at the project offset. Full coverage,
  20 frames each. The gain axis goes to 600 because that is where this camera's gain control
  actually ends - a retired project assumed 400 and silently lost a fifth of the range.
- **the fine block** re-samples gain 180 to 220 in steps of **2**. This is the only place the
  grid is dense, because it is the only place a *discontinuity* was expected, and a step change
  found on a grid of 10 is a step change you cannot locate to better than 10.
- **the offset arm** walks the offset from 0 to 50 in steps of 5 at eight gains, 10 frames each.
  Fewer frames per point, because this block answers a yes/no question (does it clip) rather
  than measuring a width.

Note what is *not* here: no exposure axis, no temperature axis. A bias frame has no exposure
worth varying, and temperature is held fixed by the project's first-pass simplification. A sweep
that measures nothing new is bench time spent, which is the whole reason the axes are argued
over before the camera is opened.

In [ ]:
block = np.where(sweep.offset != PROJECT_OFFSET, "offset arm",
                 np.where(sweep.gain % 10 != 0, "fine", "coarse"))
grid = sweep.assign(block=block)

fig, ax = plt.subplots(figsize=(7.2, 2.8))
for name, marker, colour in (("coarse", "o", "0.35"), ("fine", "s", "crimson"),
                             ("offset arm", "^", "steelblue")):
    d = grid[grid.block == name]
    ax.plot(d.gain, d.offset, marker, ms=3.5, color=colour, label=f"{name} ({len(d)})")
ax.set(xlabel="gain (ZWO units, 0.1 dB each)", ylabel="offset",
       title=f"{len(grid)} settings, and the shape is the argument")
ax.legend(fontsize=7)
fig.tight_layout()

print(grid.groupby("block").agg(settings=("gain", "size"),
                                gain_lo=("gain", "min"), gain_hi=("gain", "max"),
                                offsets=("offset", "nunique")).to_string())

## 2. The pedestal: what it is and why it is not zero

A bias frame is a read of the sensor with no exposure. In an ideal world every pixel would read
0. They read 63 counts at gain 0 and 1,035 at gain 600 instead, and that is **on purpose**.

The reason is that noise is symmetric. A pixel whose true value is 0 will, half the time, want to
report a negative number - and there is no negative number in an unsigned integer. Those values
would be silently truncated to 0, and the truncation eats the low half of the distribution: the
mean shifts up, the measured noise shrinks, and both errors are invisible in the data. So the
camera adds a deliberate positive offset, the **pedestal**, to keep the whole distribution above
the floor.

Two separate things make up that pedestal, and session 01 separates them:

- an **analogue** part, which is a small voltage present before the amplifier. Turn the gain up
  and it gets amplified along with everything else, so it grows;
- a **digital** part, added after the conversion. It is the `OFFSET` control, and it is just an
  integer added to the result, so gain does nothing to it.

The model is `pedestal = A + B * amplification`, with `amplification = 10 ** (gain / 200)`
because ZWO's gain unit is 0.1 dB - so gain 200 means 20 dB, a factor of 10 in voltage. `A` is
the digital part and `B * amplification` is the analogue part.

The left panel is that growth. The right panel is the digital part isolated: at fixed gain, the
pedestal against the offset setting is a straight line of slope **4.0032 counts per offset unit**
- and crucially it is the *same* slope at every gain, which is what proves the offset is added
after the amplifier rather than before it.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.0))

ax[0].plot(at15.gain, at15.pedestal, ".-", ms=4, lw=0.8, color="0.3")
ax[0].axvline(HCG, color="crimson", lw=0.8, ls="--")
ax[0].annotate(f"HCG at gain {HCG}", (HCG, at15.pedestal.max() * 0.8),
               xytext=(HCG + 40, at15.pedestal.max() * 0.85), fontsize=7, color="crimson")
ax[0].set(xlabel="gain", ylabel="pedestal, ADC counts",
          title=f"pedestal vs gain, at offset {PROJECT_OFFSET}")

arm = sweep[sweep.offset != PROJECT_OFFSET]
for g, d in arm.groupby("gain"):
    d = d.sort_values("offset")
    ax[1].plot(d.offset, d.pedestal, ".-", ms=4, lw=0.8, label=f"gain {g}")
ax[1].set(xlabel="offset setting", ylabel="pedestal, ADC counts",
          title="pedestal vs offset: parallel lines")
ax[1].legend(fontsize=6, ncol=2)
fig.tight_layout()

slopes = {int(g): float(np.polyfit(d.offset, d.pedestal, 1)[0])
          for g, d in arm.groupby("gain") if len(d) > 2}
print("counts of pedestal per unit of offset, measured at each gain:")
print(pd.Series(slopes).round(3).to_string())
k = K["pedestal_per_offset_unit"]
print(f"\npublished: {k['value']} +/- {k['uncertainty']} {k['unit']}")
print("The spread across gains is the evidence that the offset is digital: an analogue\n"
      "offset would be amplified, and these slopes would fan out with gain instead of\n"
      "staying parallel.")

## 3. Two branches, and why fitting through them is a real error

The kink in the left panel above is not noise. At gain 200 the sensor switches its **conversion
gain** - physically, it changes the capacitance the collected charge is dumped onto, so the same
number of electrons produces a bigger voltage. Everything downstream of that switch changes at
once: the read noise steps down, and the analogue part of the pedestal steps down with it.

That means there is no single `B` that describes both sides. The sensor has two states, and a fit
that spans them is fitting a model that the hardware does not obey.

`00` section 11 is the general lesson - read residuals as a shape - and this is the instance. The
left panel below fits one line across all 77 gains. Its residuals are not scatter; they are a
smooth arc, which is the signature of the wrong model. The right panel fits the same points in
two pieces split at gain 200, and the arc disappears.

**The residuals are plotted as percentages, and that is not cosmetic.** In absolute counts the
per-branch fit looks *worse* at its extreme - 16 counts against 13 - purely because the pedestal
itself runs from 63 to 1035 counts and the largest residual naturally lands where the pedestal is
largest. As a fraction of what it is predicting, which is how a subtraction error actually
propagates, the single fit is wrong by up to **14.5%** and the branch fits by **0.8%** (low gain)
and **2.4%** (high gain, at the very end of the exponential). Choosing the wrong denominator here
would have reversed the conclusion.

The cost of getting this wrong is not academic. A pedestal is subtracted from every dark, every
flat and every light this project will ever calibrate. **Mispredicting it by 8-17%, which is what
the single fit does, puts that error into every calibrated frame** - and it is a *bias*, not a
noise, so stacking a hundred frames does not reduce it at all.

In [ ]:
amp = 10 ** (at15.gain / 200.0)


def fit(x, y):
    B, A = np.polyfit(x, y, 1)
    resid_pct = 100 * (y - (A + B * x)) / y
    return A, B, resid_pct


fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.0), sharey=True)

A, B, resid = fit(amp, at15.pedestal)
ax[0].plot(at15.gain, resid, "o", ms=3, color="crimson")
ax[0].set(title=f"one fit through both branches\nA={A:.1f}  B={B:.2f}  "
                f"worst {resid.abs().max():.1f}%",
          xlabel="gain", ylabel="residual, % of the pedestal")

for name, mask, colour in (("lcg", at15.gain < HCG, "0.35"),
                           ("hcg", at15.gain >= HCG, "steelblue")):
    A, B, r = fit(amp[mask], at15.pedestal[mask])
    ax[1].plot(at15.gain[mask], r, "o", ms=3, color=colour,
               label=f"{name}: A={A:.1f}  B={B:.3f}  worst {r.abs().max():.2f}%")
ax[1].legend(fontsize=6)
ax[1].set(title=f"split at gain {HCG}", xlabel="gain")
for a in ax:
    a.axhline(0, color="0.6", lw=0.8)
fig.tight_layout()

print("published fit (bias_constants.json -> pedestal_fit):")
print(json.dumps(K["pedestal_fit"]["value"], indent=2))
print(f"uncertainty (worst residual, counts): {K['pedestal_fit']['uncertainty']}")
print("\nB falls {:.1f} -> {:.2f} across the transition -- a factor of {:.1f}.  That is the\n"
      "conversion-gain switch seen in the pedestal, and it is an independent fingerprint of\n"
      "the same event section 4 finds in the read noise."
      .format(K["pedestal_fit"]["value"]["lcg"]["B"], K["pedestal_fit"]["value"]["hcg"]["B"],
              K["pedestal_fit"]["value"]["lcg"]["B"] / K["pedestal_fit"]["value"]["hcg"]["B"]))

## 4. Read noise, and the cliff at gain 200

**Read noise is the price of asking the question.** Every time the sensor reads a pixel, the
amplifier and the converter add a bit of randomness that has nothing to do with light. It is
there in a 1-second exposure and in a 10-minute one, identically.

That last sentence is the whole reason this project exists. Every other noise source in the SNR
model grows with exposure time: more sky, more dark current, more photons all mean more noise as
well as more signal. **Read noise does not.** So it is the only term a longer sub-exposure
*dilutes* - it gets paid once per frame instead of once per second - and the question "how long
should a sub be?" is, underneath, the question "how much read noise am I paying per frame?".

It is measured here from the difference of two consecutive frames, divided by sqrt(2)
(`00` sections 3 and 5), which cancels every fixed feature and needs no fit.

Read the curve left to right and it does what you would expect: turn the gain up, the amplifier
amplifies the noise, `R` in counts climbs steadily from 0.66 to 76.9. Then at gain 200 it falls
**off a cliff** - from 3.59 counts at gain 198 to 1.08 at gain 200, a drop of **70% in one step
of 2**, where the typical step either side of it is under 3% - and then starts climbing again.

That is the HCG switch, and it is close to a free lunch. The sensor has changed how many counts
one electron produces, so the *same* electrons now land further apart on the scale, and the
read noise - which is added after that point - is a smaller fraction of them. The published
threshold is **gain 200 +/- 2**, where the uncertainty is the step of the fine grid: the sweep
cannot locate a discontinuity more precisely than the spacing of the points it looked at.

In [ ]:
r = at15.set_index("gain").R_sd
fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.0))

ax[0].plot(r.index, r.values, ".-", ms=4, lw=0.8, color="0.3")
ax[0].axvline(HCG, color="crimson", lw=0.8, ls="--")
ax[0].set(yscale="log", xlabel="gain", ylabel="R, ADC counts",
          title="read noise across the whole gain range (log scale)")

near = at15[(at15.gain >= 150) & (at15.gain <= 260)]
ax[1].plot(near.gain, near.R_sd, "o-", ms=4, lw=0.8, color="0.3")
ax[1].errorbar(near.gain, near.R_sd, yerr=near.R_err, fmt="none", ecolor="crimson", lw=1)
ax[1].axvline(HCG, color="crimson", lw=0.8, ls="--")
ax[1].set(xlabel="gain", ylabel="R, ADC counts",
          title="the fine grid, error bars included\n(they are smaller than the dots)")
fig.tight_layout()

step = (r.diff() / r.shift()).rename("fractional change")
worst = step.loc[178:222]
print("fractional change in R between adjacent gains, over the fine grid:")
print(worst.round(4).to_string())
print(f"\nlargest drop {step.min():+.1%} entering gain {int(step.idxmin())}")
print(f"typical step elsewhere {step.drop(step.idxmin()).abs().median():+.1%}")
print(f"\npublished: HCG threshold {HCG} +/- {K['hcg_threshold_gain']['uncertainty']}, "
      f"R there {K['read_noise_at_hcg']['value']} +/- "
      f"{K['read_noise_at_hcg']['uncertainty']} counts")

### What the cliff is worth, and what it costs

The temptation is to read "lower read noise" as "better" and stop there. It is not that simple,
and the trade is worth stating plainly even though this session cannot price it fully.

Raising the gain divides the full well - the largest signal a pixel can record before the number
stops going up - by the same factor it multiplies everything else. At gain 0 this camera can
record a very bright star; at gain 400 it cannot. So gain buys lower read noise **in counts** and
sells dynamic range, and the usual answer is "somewhere in the middle".

The HCG step is the exception, and that is why it is worth locating precisely: across it, `R`
drops by a factor of three *without* a corresponding step in the well. It is a genuine
discontinuity in the trade, not a point on the smooth curve.

**This notebook cannot finish that argument**, and it is worth being clear about why. Full well
and dynamic range are quantities in electrons; converting counts to electrons needs `g`; `g` is
the photon transfer curve. What session 01 establishes is *where* the discontinuity is and how
big it is in counts. What it is worth in electrons is session 03's question.

## 5. Against ZWO's published curves

ZWO publish a read-noise-versus-gain chart for this camera. `vendor/asi585specs/gain-curves.csv`
is that chart read off by eye into a table, and the project's rule is that a vendor number is a
**hypothesis** - something to reproduce, never to import. Read off a plot, it is worth +/-5% at
best.

The comparison can only run in one direction, and the reason is the unit. ZWO quote read noise in
**electrons**; this session works in **counts**. Converting our measurement into electrons needs
`g`, which we do not have. So instead the *prediction* is converted into our units, using ZWO's
own `g` curve: `R_predicted_counts = R_e / g`.

That makes this a **joint test of two published curves at once**, and it is important to say so.
If the ratio comes out away from 1, the disagreement could be in their read-noise curve, in their
gain curve, or in ours - and nothing in this session can separate those. Session 03 measures `g`
independently, and that is what turns this into two separate tests.

What we can already say: the shape matches. The cliff is where they annotate it, the curve rises
with the same slope on both branches, and from gain 0 to 300 the ratio stays inside 0.94 to 1.05
- exactly the margin that reading numbers off a printed chart is worth.

Above that it drifts: 0.85 at gain 350, 1.27 at gain 450. Both sit where their chart is steepest
and hardest to read by eye, and where a hand-digitised curve is least trustworthy. It is not yet
evidence of anything about the sensor.

In [ ]:
spec = pd.read_csv(SPECS)
spec = spec[spec.branch.values == np.where(spec.gain >= HCG, "hcg", "lcg")]
cmp = (spec.set_index("gain")[["read_noise_e", "g_e_per_adu", "read_noise_adu_predicted"]]
       .join(at15.set_index("gain")[["R_sd", "R_err"]]).dropna())
cmp["ratio"] = cmp.R_sd / cmp.read_noise_adu_predicted

fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.0))
ax[0].plot(at15.gain, at15.R_sd, "-", lw=1, color="0.3", label="measured (this session)")
ax[0].plot(cmp.index, cmp.read_noise_adu_predicted, "o", ms=4, color="crimson",
           label="ZWO, converted with their own g")
ax[0].set(yscale="log", xlabel="gain", ylabel="R, ADC counts", title="measured vs predicted")
ax[0].legend(fontsize=7)

ax[1].plot(cmp.index, cmp.ratio, "o-", ms=4, lw=0.8, color="crimson")
ax[1].axhline(1, color="0.6", lw=0.8)
ax[1].axhspan(0.95, 1.05, color="0.85", zorder=0)
ax[1].set(xlabel="gain", ylabel="measured / predicted",
          title="the ratio; the band is the +/-5% a chart read-off is worth")
fig.tight_layout()

print(cmp.round(3).to_string())
print(f"\nratio: {cmp.ratio.min():.2f} to {cmp.ratio.max():.2f}, median {cmp.ratio.median():.2f}")
print("\nZWO's chart stops at gain 450; this camera's control runs to 600, so the last\n"
      "quarter of the measured curve has no published counterpart at all.")

## 6. Choosing the offset - the most instructive argument in the session

Section 2 said the pedestal exists to keep the distribution off the zero floor. So: how much
pedestal is enough? Too little and the data is quietly truncated. Too much and full scale is
wasted on empty pedestal, for nothing.

The obvious test - "are any pixels reading zero?" - is wrong, and `00` section 10 is why: the
minimum of a quarter of a million samples sits about 4.9 sigma below centre **by chance**, so a
few zeros are expected even when nothing is wrong. A retired project acted on exactly this and
condemned a perfectly good offset over 58 pixels in a million.

The right test is a **fraction**, against a threshold of 0.1%. That is an estimator: it
converges, it can be compared against a prediction, and it does not care how many pixels you
looked at.

**And the sweep found that the fraction is still not enough.** Two completely different things
make a pixel read zero:

- **clipping** - the distribution genuinely reaches the floor. Raise the offset and it goes away.
- **telegraph pixels** - individual pixels that randomly switch to a low state regardless of how
  much headroom they have. Raise the offset and they are still there.

They look identical in a fraction. So the published criterion is a pair of tests: the fraction
under 0.1% **and** the pedestal at least **15 R** above zero. Fifteen read noises is not
caution - by the Gaussian tail (`00` section 9) it is a guarantee of about 3.7e-51, which is to
say never. And the floor was not picked, it was **bracketed**: offset 5 still clips at 13.5 R,
offset 10 does not at 19.6 R, so the boundary lies between them and the arm's step of 5 is the
uncertainty on where.

In [ ]:
judged = sweep[(sweep.gain <= 300) & sweep.zero_frac.notna()].copy()

fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.0))
for g, d in judged.groupby("gain"):
    d = d.sort_values("offset")
    ax[0].plot(d.offset, d.zero_frac.clip(lower=1e-9), ".-", ms=4, lw=0.8, label=f"gain {g}")
    ax[1].plot(d.offset, d.sigmas_above_zero, ".-", ms=4, lw=0.8)
ax[0].axhline(0.001, color="crimson", lw=1, ls="--")
ax[0].text(30, 0.0013, "0.1% threshold", color="crimson", fontsize=7)
ax[0].set(yscale="log", xlabel="offset", ylabel="fraction of pixels at zero",
          title="test 1: the clipped fraction")
ax[0].legend(fontsize=6, ncol=2)
ax[1].axhline(15, color="crimson", lw=1, ls="--")
ax[1].text(30, 17, "15 R floor", color="crimson", fontsize=7)
ax[1].set(yscale="log", xlabel="offset", ylabel="pedestal / R",
          title="test 2: headroom, in read noises")
fig.tight_layout()

safe = judged.assign(ok=(judged.zero_frac < 0.001) & (judged.sigmas_above_zero >= 15))
per_offset = safe.groupby("offset").ok.all()
print("offsets passing both tests at every judged gain (0-300):")
print(per_offset.to_string())
print(f"\npublished: offset_min_safe = {K['offset_min_safe']['value']} "
      f"+/- {K['offset_min_safe']['uncertainty']}, "
      f"project_offset = {K['project_offset']['value']}")

### Why the project fixes on 15 rather than the 10 it could defend

Ten is the smallest offset the evidence supports. The project uses 15, and the reasoning is
worth reading because it is a *decision* rather than a measurement - which is why its published
uncertainty is 0, a field that would otherwise look like a mistake.

Five extra offset units cost **20 counts of pedestal**, which is 0.49% of full scale. What they
buy is margin against the fact that 10 is the *edge* of the safe region, plus something with no
statistical content at all: **the 15,090 frames in the historic archive were all shot at offset
15**. Fixing on 15 makes every one of those frames directly comparable with everything this
project measures from now on. Half a per cent of dynamic range is a cheap price for a year of
existing data.

### The gain-600 exclusion, which is where the pixels have to be consulted

One setting refuses to behave: gain 600 has pixels reading zero at **every** offset in the arm,
including offsets where the pedestal sits over a thousand counts clear of the floor. Clipping is
arithmetically impossible there. Left in the criterion it does not produce a
contradiction, which would at least be obvious - it quietly drags the answer up to **offset
45**, the point at which gain 600's telegraph pixels happen to clear an arbitrary headroom line.
That is a threshold answering a question about defective pixels while wearing the label of an
answer about clipping.

So which is it - dead pixels, or telegraph pixels? A fraction cannot say. The test that can needs
the pixels themselves: **are they the same pixels at both offsets?** Dead pixels are the same
pixels every time. Telegraph pixels are a different random subset in every frame.

In [ ]:
if HAVE_FRAMES:
    lo = ST.to_adc(F.read(FRAMES / "offsetarm_g600_o000_005.fits")[0]) == 0
    hi = ST.to_adc(F.read(FRAMES / "offsetarm_g600_o050_005.fits")[0]) == 0
    both = int((lo & hi).sum())
    print(f"gain 600, offset 0:  {int(lo.sum()):>5} pixels read zero")
    print(f"gain 600, offset 50: {int(hi.sum()):>5} pixels read zero")
    print(f"in both:             {both:>5}  ({both / max(int(hi.sum()), 1):.0%} overlap)")
    print("\nDead pixels would overlap almost completely.  These barely overlap at all, so\n"
          "they are telegraph pixels swinging low, and no offset buys them away.")

    fig, ax = plt.subplots(1, 2, figsize=(7.4, 3.6))
    for a, mask, t in ((ax[0], lo, "offset 0"), (ax[1], hi, "offset 50")):
        ys, xs = np.nonzero(mask)
        a.plot(xs, ys, ".", ms=1.5, color="crimson")
        a.set(title=f"{t}: {mask.sum()} zero-valued pixels", xlim=(0, mask.shape[1]),
              ylim=(0, mask.shape[0]))
        a.set_xticks([]); a.set_yticks([])
    fig.suptitle("gain 600: where the zeros are, at two offsets", fontsize=9)
    fig.tight_layout()
else:
    print("frames absent -- see 03_bias_sweep.ipynb, which records 11% overlap.")

g600 = sweep[(sweep.gain == 600)][["offset", "pedestal", "R_at_offset",
                                   "sigmas_above_zero", "zero_frac"]]
print("\ngain 600 across the offset arm -- the floor survives every offset:")
print(g600.sort_values("offset").round(4).to_string(index=False))

## 7. No fixed pattern: the finding that saves the most work

A **master bias** - many bias frames averaged together, subtracted from every science frame - is
standard practice, and its purpose is to remove *structure*: pixels that are reliably a little
brighter or darker than their neighbours, columns with their own offset, amplifier glow. If that
structure exists, only a per-pixel map removes it.

Session 01 measured whether it exists here, with the comparison `00` section 4 sets up: the
spread **across pixels** in one frame against the spread **across time** at one pixel. If the
sensor has fixed structure, the spatial spread is the larger of the two, because it contains
the noise *and* the pattern.

The answer is **1.011 +/- 0.007** across 77 gains. They are the same number. There is no
structure, and therefore nothing for a per-pixel master bias to remove that a single scalar - the
pedestal - does not already handle.

Two consequences, one practical and one methodological:

- **the bias model is one number per (gain, offset).** No master bias frames to shoot, store,
  match or worry about going stale.
- **it licenses `R` as the yardstick in section 6.** "15 read noises of headroom" is only
  meaningful if the thing being truncated has width `R`. What the floor actually truncates is the
  pixel-to-pixel distribution, so the argument needs those two widths to be the same - and here
  they are, by measurement rather than by assumption.

The rise at the very top of the range is real and worth noting: by gain 600 the ratio reaches
1.02, which is the telegraph pixels of section 6 showing up in the spatial spread.

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 2.8))
ax.plot(at15.gain, at15.fpn_ratio, ".-", ms=4, lw=0.8, color="0.3")
ax.axhline(1, color="crimson", lw=0.8, ls="--")
ax.axvline(HCG, color="steelblue", lw=0.8, ls=":")
ax.set(xlabel="gain", ylabel="spatial sd / temporal R",
       title="fixed-pattern ratio: flat, and flat at 1")
fig.tight_layout()

k = K["bias_fixed_pattern_ratio"]
print(f"published: {k['value']} +/- {k['uncertainty']} ({k['unit']})")
print(f"measured range over the sweep: {at15.fpn_ratio.min():.4f} to {at15.fpn_ratio.max():.4f}")
print(f"  below the HCG threshold: {at15[at15.gain < HCG].fpn_ratio.mean():.4f}")
print(f"  at gain 600:             {float(at15[at15.gain == 600].fpn_ratio.iloc[0]):.4f}")

## 8. Does the pedestal drift? The question session 02 asked

Everything above treats the pedestal as a fixed property of a setting. If it were slowly moving
instead, then a bias frame taken at the start of a night would be the wrong bias frame by the end
of it - and subtracting it would leave a residual that looks exactly like dark current. That is
not hypothetical: L14 records a master dark landing *below* its master bias, producing a negative
dark current, because the two were shot four hours apart.

So session 01 shot a dedicated trace: 450 frames, one every 2 seconds for 15 minutes, everything
else held fixed.

The fitted slope is -0.00133 counts per minute. **That number on its own means nothing**, and
`00` section 12 is the reason: fit a line to noise and you always get a slope. What decides the
question is the slope against its own uncertainty, which is 0.0028 counts/min - so the fitted
drift is under half of one error bar, and the honest statement is a **bound**:

> No pedestal drift above 0.006 counts/minute over 15 minutes at -10 C, which is under 0.1
> counts across the whole trace.

For scale, `R` at this gain is 1.43 counts. A drift of 0.08 counts over a quarter of an hour is
under 6% of a single read noise - far below anything that could contaminate a dark frame.

**So interleaving bias frames between darks in session 02 is a precaution, not a requirement.**
It stays in the protocol, cheaply, because 15 minutes at one temperature is not a statement about
four hours across a night - but nothing in session 02's design has to be built around it.

In [ ]:
n = len(drift)
slope, intercept = np.polyfit(drift.elapsed_s, drift.pedestal, 1)
resid = drift.pedestal - (slope * drift.elapsed_s + intercept)
se = resid.std(ddof=2) / (drift.elapsed_s.std(ddof=1) * math.sqrt(n - 1))

fig, ax = plt.subplots(2, 1, figsize=(7.2, 4.0), sharex=True,
                       gridspec_kw={"height_ratios": [2, 1]})
ax[0].plot(drift.elapsed_s / 60, drift.pedestal, ".", ms=2, color="0.55")
ax[0].plot(drift.elapsed_s / 60, slope * drift.elapsed_s + intercept, color="crimson", lw=1.2,
           label=f"fit {slope * 60:+.4f} +/- {se * 60:.4f} counts/min")
ax[0].set(ylabel="pedestal, ADC counts", title="450 bias frames over 15 minutes, gain 100")
ax[0].legend(fontsize=7)
ax[1].plot(drift.elapsed_s / 60, drift.ccd_temp, ".", ms=2, color="steelblue")
ax[1].set(xlabel="minutes", ylabel="sensor, C", ylim=(-11, -9))
fig.tight_layout()

print(f"slope                  {slope * 60:+.5f} counts/min")
print(f"its standard error     {se * 60:.5f} counts/min")
print(f"slope / error          {slope / se:+.2f}   -- under half an error bar: no detection")
print(f"2-sigma bound          |rate| < {2 * se * 60:.4f} counts/min "
      f"= {2 * se * 60 * 15:.3f} counts over the trace")
print(f"R at this gain         {float(at15[at15.gain == 100].R_sd.iloc[0]):.3f} counts")
print(f"temperature held       {drift.ccd_temp.min()} to {drift.ccd_temp.max()} C")

## 9. What the session settled, and what it did not

**Settled, and available to every later notebook:**

| constant | value | what it unlocks |
|---|---|---|
| `pedestal_fit` | `A + B * 10**(gain/200)`, per branch | the pedestal at any gain, without re-shooting bias frames |
| `pedestal_per_offset_unit` | 4.0032 counts/unit | the offset is digital, so the axis is retired |
| `hcg_threshold_gain` | 200 +/- 2 | where the free read-noise step lives |
| `read_noise_at_hcg` | 1.0768 +/- 0.0005 counts | the `R^2 / t` term of the SNR model, in counts |
| `project_offset` | 15 | fixed for the life of the project |
| `bias_fixed_pattern_ratio` | 1.011 | no master bias needed |
| `pedestal_drift_rate` | consistent with zero | session 02 need not be built around interleaving |

**The offset axis is now closed.** `A` scales linearly with offset, and `R` was independent of
offset to better than 0.5% everywhere it was not clipping. Two axes went into this sweep and one
came out; the offset never has to be swept again.

**Not settled, and deliberately so:**

- **Nothing is in electrons.** Every number above is in ADC counts. The conversion is `g`, the
  system gain, and measuring it is the photon transfer curve - session 03. Until then "1.08
  counts of read noise" cannot be compared with any published figure for any other camera.
- **Full well, dynamic range and the actual gain recommendation** all wait on the same `g`.
  Session 01 located the HCG step; it did not price it.
- **The vendor comparison is still joint.** Section 5 tests ZWO's read-noise curve and their gain
  curve together. Only an independent `g` separates them.
- **Nothing here is a statement about temperature.** Everything was measured at -10 C, and the
  model's first pass treats temperature as fixed rather than as an axis.

**What comes next.** `05_dark_bound` reads session 02's interleaved darks for the dark-current
term `D` at -10 C - an upper bound if that is what the data supports, quoted as one - and
measures how a real stack of frames approaches the ideal sqrt(N), which is the `eta_comb` term.
Both of those lean on the constants above: the dark measurement subtracts the pedestal this
session fitted, and it uses this session's `R` to know what "below the noise" means.